# 04 - Forecast FinOps de costos cloud con modelo de serie de tiempo

## Caso fintech
Este notebook simula costos cloud diarios de una fintech y predice el gasto futuro.

## Objetivo
Construir un modelo de forecasting para estimar costos cloud.

## Dataset
Se genera un dataset sintético de 1,095 días, equivalente a 3 años de costos diarios.

## Técnicas
- Series de tiempo
- Tendencia
- Estacionalidad semanal y mensual
- Rolling averages
- SARIMAX
- Métricas MAE y RMSE

Nota: se usa SARIMAX porque viene en `statsmodels`. Puedes reemplazarlo luego por Prophet si lo instalas.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)


In [ ]:
days = 1095
dates = pd.date_range(start="2023-01-01", periods=days, freq="D")

trend = np.linspace(1800, 4200, days)
weekly_seasonality = 250 * np.sin(2 * np.pi * np.arange(days) / 7)
monthly_seasonality = 400 * np.sin(2 * np.pi * np.arange(days) / 30)
noise = np.random.normal(0, 180, days)

# Eventos de incremento: campañas, migraciones, alta demanda
campaign_effect = np.zeros(days)
campaign_effect[250:280] += 900
campaign_effect[610:650] += 1200
campaign_effect[900:940] += 800

cloud_cost = trend + weekly_seasonality + monthly_seasonality + noise + campaign_effect
cloud_cost = np.clip(cloud_cost, 800, None)

df = pd.DataFrame({
    "date": dates,
    "cloud_cost_usd": cloud_cost.round(2)
})

df.head()


In [ ]:
df.shape

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df["date"], df["cloud_cost_usd"])
plt.title("Costo cloud diario simulado")
plt.xlabel("Fecha")
plt.ylabel("USD")
plt.show()


In [ ]:
df["rolling_7d"] = df["cloud_cost_usd"].rolling(7).mean()
df["rolling_30d"] = df["cloud_cost_usd"].rolling(30).mean()

plt.figure(figsize=(12, 5))
plt.plot(df["date"], df["cloud_cost_usd"], alpha=0.4, label="Daily cost")
plt.plot(df["date"], df["rolling_7d"], label="Rolling 7d")
plt.plot(df["date"], df["rolling_30d"], label="Rolling 30d")
plt.title("Costo cloud con medias móviles")
plt.xlabel("Fecha")
plt.ylabel("USD")
plt.legend()
plt.show()


In [ ]:
series = df.set_index("date")["cloud_cost_usd"]

train = series.iloc[:-90]
test = series.iloc[-90:]

model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

results = model.fit(disp=False)
forecast = results.forecast(steps=len(test))


In [ ]:
mae = mean_absolute_error(test, forecast)
rmse = mean_squared_error(test, forecast, squared=False)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train.index, train, label="Train")
plt.plot(test.index, test, label="Actual")
plt.plot(test.index, forecast, label="Forecast")
plt.title("Forecast de costos cloud")
plt.xlabel("Fecha")
plt.ylabel("USD")
plt.legend()
plt.show()


In [ ]:
future_forecast = results.forecast(steps=120)

future_df = pd.DataFrame({
    "date": future_forecast.index,
    "forecast_cloud_cost_usd": future_forecast.values.round(2)
})

future_df.head()


In [ ]:
monthly_projection = future_df.set_index("date").resample("M")["forecast_cloud_cost_usd"].sum().round(2)
monthly_projection